# 69 - STRAT-003 v3: Checkmate Signal Position Sizing

**Finding from notebook 68:**
- Checkmate signal doesn't help STRAT-002 (already very selective)
- Checkmate signal DOES help STRAT-003 via **position sizing** (+15.7% Sharpe)

**This notebook:**
1. Deep dive into optimal sizing thresholds
2. Test different sizing curves (linear, stepped, aggressive)
3. Analyze trade-by-trade impact
4. Walk-forward validation
5. Final v3 specification

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)

DATA_DIR = Path.home() / "Documents" / "bitcoin-lab-btc-data-pipeline" / "data" / "daily"

def load_metric(name):
    path = DATA_DIR / f"{name}.parquet"
    if not path.exists():
        print(f"  Warning: {name} not found")
        return pd.DataFrame(columns=['time', 'value'])
    df = pd.read_parquet(path)
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        if df['time'].dt.tz is not None:
            df['time'] = df['time'].dt.tz_localize(None)
        df = df.set_index('time').sort_index()
    return df

# Load all needed metrics
metrics = ['price', 'sopr', 'sopr_sth', 'sopr_lth', 'mvrv', 'mvrv_sth', 'mvrv_lth', 'nupl', 'aviv']
data = {m: load_metric(m) for m in metrics}
print("Data loaded")

In [ ]:
# Build master dataframe
df = data['price'][['value']].rename(columns={'value': 'price'}).copy()
df['returns'] = df['price'].pct_change()

# Add metrics
metric_map = {
    'sopr': 'sopr', 'sopr_sth': 'sth_sopr', 'sopr_lth': 'lth_sopr',
    'mvrv': 'mvrv', 'mvrv_sth': 'mvrv_sth', 'mvrv_lth': 'mvrv_lth',
    'nupl': 'nupl', 'aviv': 'aviv'
}

for src, dst in metric_map.items():
    if src in data and not data[src].empty:
        df = df.join(data[src][['value']].rename(columns={'value': dst}), how='left')
        df[dst] = df[dst].ffill()

# Build Checkmate composite signal
SIGNAL_CONFIG = {
    'mvrv': {'weight': 0.30, 'bullish': 1.0, 'bearish': 2.4},
    'mvrv_sth': {'weight': 0.15, 'bullish': 1.0, 'bearish': 1.4},
    'mvrv_lth': {'weight': 0.15, 'bullish': 1.5, 'bearish': 3.5},
    'nupl': {'weight': 0.20, 'bullish': 0.25, 'bearish': 0.6},
    'sopr': {'weight': 0.10, 'bullish': 1.0, 'bearish': 1.05},
    'aviv': {'weight': 0.10, 'bullish': 1.0, 'bearish': 1.5},
}

def score_metric(value, bullish, bearish):
    if pd.isna(value): return 0
    midpoint = (bullish + bearish) / 2
    if value <= bullish:
        return -1 - (bullish - value) / bullish
    elif value <= midpoint:
        return -1 + (value - bullish) / (midpoint - bullish)
    elif value <= bearish:
        return (value - midpoint) / (bearish - midpoint)
    else:
        return 1 + min((value - bearish) / bearish, 1)

def calc_checkmate_signal(row):
    total_score, total_weight = 0, 0
    for metric, cfg in SIGNAL_CONFIG.items():
        if metric in row and pd.notna(row[metric]):
            total_score += score_metric(row[metric], cfg['bullish'], cfg['bearish']) * cfg['weight']
            total_weight += cfg['weight']
    return total_score / total_weight if total_weight > 0 else 0

df['checkmate'] = df.apply(calc_checkmate_signal, axis=1)

# STRAT-003 entry signal
df['entry_raw'] = df['sth_sopr'] < 1.0
df['entry'] = df['entry_raw'] & ~df['entry_raw'].shift(1).fillna(False)

# Exit trigger
df['exit_trigger'] = (df['mvrv'] > 2.5) & (df['lth_sopr'] > 1.5)

print(f"Data: {df.index[0].date()} to {df.index[-1].date()}")
print(f"Entries: {df['entry'].sum()}")
print(f"Checkmate range: {df['checkmate'].min():.2f} to {df['checkmate'].max():.2f}")

---
## Part 1: Define Position Sizing Functions

In [ ]:
def sizing_fixed(signal):
    """No sizing - always 100%"""
    return 1.0

def sizing_stepped_v1(signal):
    """Original stepped sizing"""
    if signal <= -1.0:   return 1.00
    elif signal <= -0.5: return 0.80
    elif signal <= 0.0:  return 0.60
    elif signal <= 0.5:  return 0.40
    else:                return 0.25

def sizing_stepped_v2(signal):
    """More aggressive at extremes"""
    if signal <= -1.5:   return 1.00
    elif signal <= -1.0: return 0.90
    elif signal <= -0.5: return 0.75
    elif signal <= 0.0:  return 0.50
    elif signal <= 0.5:  return 0.35
    else:                return 0.20

def sizing_stepped_v3(signal):
    """Conservative - higher minimum"""
    if signal <= -1.0:   return 1.00
    elif signal <= -0.5: return 0.85
    elif signal <= 0.0:  return 0.70
    elif signal <= 0.5:  return 0.55
    else:                return 0.40

def sizing_linear(signal, min_size=0.25, max_size=1.0):
    """Linear interpolation between -1.5 and +1.5"""
    # Clamp signal to [-1.5, 1.5]
    s = max(-1.5, min(1.5, signal))
    # Linear from max_size at -1.5 to min_size at +1.5
    return max_size - (s + 1.5) / 3.0 * (max_size - min_size)

def sizing_aggressive(signal):
    """Very aggressive - big swings"""
    if signal <= -1.0:   return 1.00
    elif signal <= -0.5: return 0.80
    elif signal <= 0.0:  return 0.50
    elif signal <= 0.5:  return 0.30
    else:                return 0.15

# Visualize sizing functions
signals = np.linspace(-2, 2, 100)
fig, ax = plt.subplots(figsize=(12, 6))

sizing_funcs = [
    ('Fixed 100%', sizing_fixed, 'gray'),
    ('Stepped v1', sizing_stepped_v1, '#3b82f6'),
    ('Stepped v2', sizing_stepped_v2, '#22c55e'),
    ('Stepped v3', sizing_stepped_v3, '#f59e0b'),
    ('Linear', sizing_linear, '#a855f7'),
    ('Aggressive', sizing_aggressive, '#ef4444'),
]

for name, func, color in sizing_funcs:
    sizes = [func(s) for s in signals]
    ax.plot(signals, [s*100 for s in sizes], label=name, color=color, linewidth=2)

ax.axvline(x=0, color='white', linestyle='--', alpha=0.3)
ax.axhline(y=100, color='white', linestyle='--', alpha=0.3)
ax.set_xlabel('Checkmate Signal')
ax.set_ylabel('Position Size (%)')
ax.set_title('Position Sizing Functions', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(-2, 2)
ax.set_ylim(0, 110)

plt.tight_layout()
plt.show()

---
## Part 2: Backtest All Sizing Functions

In [ ]:
def backtest_strat003(df, sizing_func, name):
    """
    Backtest STRAT-003 with specified sizing function.
    Entry: STH-SOPR < 1
    Exit: 30% trail, tightens to 15% after MVRV>2.5 + LTH-SOPR>1.5
    """
    bt = df[['price', 'returns', 'checkmate', 'entry', 'exit_trigger']].copy()
    
    in_position = False
    entry_price = 0
    peak_price = 0
    triggered = False
    position_size = 0
    positions = []
    trades = []
    
    for date, row in bt.iterrows():
        price = row['price']
        signal = row['checkmate']
        
        if not in_position:
            if row['entry']:
                in_position = True
                entry_price = price
                peak_price = price
                triggered = False
                position_size = sizing_func(signal)
                trades.append({
                    'entry_date': date, 'entry_price': price, 
                    'entry_signal': signal, 'size': position_size
                })
            positions.append(0)
        else:
            peak_price = max(peak_price, price)
            
            # Check trigger
            if not triggered and row['exit_trigger']:
                triggered = True
            
            # Trail percentage
            trail_pct = 0.15 if triggered else 0.30
            drawdown = (peak_price - price) / peak_price
            
            if drawdown >= trail_pct:
                # Exit
                in_position = False
                pnl = (price / entry_price - 1) * 100
                pnl_sized = pnl * position_size
                trades[-1].update({
                    'exit_date': date, 'exit_price': price,
                    'pnl': pnl, 'pnl_sized': pnl_sized,
                    'triggered': triggered, 'hold_days': (date - trades[-1]['entry_date']).days
                })
                positions.append(0)
            else:
                positions.append(position_size)
    
    bt['position'] = positions
    bt['position'] = bt['position'].shift(1).fillna(0)
    bt['strat_returns'] = bt['position'] * bt['returns']
    bt['equity'] = 100000 * (1 + bt['strat_returns']).cumprod()
    bt['dd'] = bt['equity'] / bt['equity'].cummax() - 1
    
    years = (bt.index[-1] - bt.index[0]).days / 365
    completed = [t for t in trades if 'exit_date' in t]
    
    return {
        'name': name,
        'total_return': (bt['equity'].iloc[-1] / 100000 - 1) * 100,
        'cagr': ((bt['equity'].iloc[-1] / 100000) ** (1/years) - 1) * 100,
        'max_dd': bt['dd'].min() * 100,
        'sharpe': (bt['strat_returns'].mean() / bt['strat_returns'].std()) * np.sqrt(365) if bt['strat_returns'].std() > 0 else 0,
        'trades': len(completed),
        'win_rate': len([t for t in completed if t['pnl'] > 0]) / len(completed) * 100 if completed else 0,
        'avg_size': np.mean([t['size'] for t in completed]) if completed else 0,
        'avg_pnl': np.mean([t['pnl'] for t in completed]) if completed else 0,
        'equity': bt['equity'],
        'dd': bt['dd'],
        'trade_log': trades
    }

In [ ]:
# Run all backtests
results = []
for name, func, _ in sizing_funcs:
    result = backtest_strat003(df, func, name)
    results.append(result)

# HODL baseline
hodl = 100000 * (1 + df['returns']).cumprod()
years = (df.index[-1] - df.index[0]).days / 365
hodl_cagr = ((hodl.iloc[-1] / 100000) ** (1/years) - 1) * 100
hodl_dd = (hodl / hodl.cummax() - 1).min() * 100
hodl_sharpe = (df['returns'].mean() / df['returns'].std()) * np.sqrt(365)

print("="*110)
print("STRAT-003 SIZING COMPARISON")
print("="*110)
print(f"\n{'Sizing':<15} {'Return':>12} {'CAGR':>8} {'MaxDD':>8} {'Sharpe':>8} {'Trades':>8} {'WinRate':>8} {'AvgSize':>8} {'AvgPnL':>8}")
print("-"*110)
print(f"{'HODL':<15} {(hodl.iloc[-1]/100000-1)*100:>11.0f}% {hodl_cagr:>7.1f}% {hodl_dd:>7.1f}% {hodl_sharpe:>8.2f} {'--':>8} {'--':>8} {'100%':>8} {'--':>8}")
print("-"*110)

for r in results:
    print(f"{r['name']:<15} {r['total_return']:>11.0f}% {r['cagr']:>7.1f}% {r['max_dd']:>7.1f}% "
          f"{r['sharpe']:>8.2f} {r['trades']:>8} {r['win_rate']:>7.0f}% {r['avg_size']:>7.0%} {r['avg_pnl']:>+7.0f}%")

print("-"*110)
best = max(results, key=lambda x: x['sharpe'])
print(f"\n🏆 Best Sharpe: {best['name']} ({best['sharpe']:.2f})")
best_return = max(results, key=lambda x: x['total_return'])
print(f"💰 Best Return: {best_return['name']} ({best_return['total_return']:.0f}%)")

In [ ]:
# Visualize equity curves
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Equity curves
axes[0].semilogy(hodl.index, hodl, 'orange', linewidth=2, alpha=0.5, label=f'HODL ({hodl_cagr:.0f}%)')
for r, (_, _, color) in zip(results, sizing_funcs):
    axes[0].semilogy(r['equity'].index, r['equity'], color=color, linewidth=1.5,
                     label=f"{r['name']} ({r['cagr']:.0f}%, Sharpe={r['sharpe']:.2f})")

axes[0].set_ylabel('Equity ($)')
axes[0].set_title('STRAT-003: Position Sizing Comparison', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper left', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Drawdowns
for r, (_, _, color) in zip(results, sizing_funcs):
    axes[1].fill_between(r['dd'].index, r['dd']*100, 0, color=color, alpha=0.3)
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 3: Trade-by-Trade Analysis

In [ ]:
# Compare trades: Fixed vs Best Sizing
fixed_result = results[0]  # Fixed 100%
best_result = best

print("\nTRADE-BY-TRADE COMPARISON: Fixed vs", best['name'])
print("="*100)

fixed_trades = [t for t in fixed_result['trade_log'] if 'exit_date' in t]
best_trades = [t for t in best_result['trade_log'] if 'exit_date' in t]

print(f"\n{'Entry Date':<12} {'Signal':>8} {'Size (F)':>10} {'Size (B)':>10} {'PnL':>10} {'PnL*Size':>10} {'Better':>8}")
print("-"*80)

for ft, bt in zip(fixed_trades, best_trades):
    signal = ft['entry_signal']
    size_f = ft['size']
    size_b = bt['size']
    pnl = ft['pnl']
    pnl_sized = bt['pnl'] * bt['size']
    
    # Who won?
    if pnl > 0:  # Winning trade
        better = 'Fixed' if size_f > size_b else 'Sized' if size_b > size_f else 'Tie'
    else:  # Losing trade
        better = 'Sized' if size_f > size_b else 'Fixed' if size_b > size_f else 'Tie'
    
    print(f"{ft['entry_date'].strftime('%Y-%m-%d'):<12} {signal:>+7.2f} {size_f:>9.0%} {size_b:>9.0%} "
          f"{pnl:>+9.0f}% {pnl_sized:>+9.0f}% {better:>8}")

In [ ]:
# Analyze: Does sizing help on winning vs losing trades?
print("\nSIZING IMPACT ANALYSIS")
print("="*60)

winners = [t for t in best_trades if t['pnl'] > 0]
losers = [t for t in best_trades if t['pnl'] <= 0]

print(f"\nWinning Trades ({len(winners)}):")
if winners:
    print(f"  Avg Signal at Entry: {np.mean([t['entry_signal'] for t in winners]):+.2f}")
    print(f"  Avg Size: {np.mean([t['size'] for t in winners]):.0%}")
    print(f"  Avg PnL: {np.mean([t['pnl'] for t in winners]):+.0f}%")

print(f"\nLosing Trades ({len(losers)}):")
if losers:
    print(f"  Avg Signal at Entry: {np.mean([t['entry_signal'] for t in losers]):+.2f}")
    print(f"  Avg Size: {np.mean([t['size'] for t in losers]):.0%}")
    print(f"  Avg PnL: {np.mean([t['pnl'] for t in losers]):+.0f}%")

# Key insight
if winners and losers:
    win_signal = np.mean([t['entry_signal'] for t in winners])
    lose_signal = np.mean([t['entry_signal'] for t in losers])
    print(f"\n💡 Insight: Winners entered at signal {win_signal:+.2f}, losers at {lose_signal:+.2f}")
    if win_signal < lose_signal:
        print("   → Lower signal (more bullish) = better trades!")
        print("   → Sizing correctly gives MORE to winners, LESS to losers!")

---
## Part 4: Optimize Sizing Thresholds

In [ ]:
# Grid search over sizing parameters
def make_stepped_sizing(thresholds, sizes):
    """Create a stepped sizing function from thresholds and sizes."""
    def sizing(signal):
        for thresh, size in zip(thresholds, sizes):
            if signal <= thresh:
                return size
        return sizes[-1]
    return sizing

# Test different threshold/size combinations
grid_results = []

# Vary the thresholds
threshold_sets = [
    ([-1.0, -0.5, 0.0, 0.5], [1.0, 0.8, 0.6, 0.4, 0.25]),  # Original
    ([-1.5, -0.75, 0.0, 0.75], [1.0, 0.8, 0.6, 0.4, 0.2]),  # Wider
    ([-0.75, -0.25, 0.25, 0.75], [1.0, 0.8, 0.6, 0.4, 0.25]),  # Narrower
    ([-1.0, -0.5, 0.0, 0.5], [1.0, 0.9, 0.7, 0.5, 0.3]),  # Higher min
    ([-1.0, -0.5, 0.0, 0.5], [1.0, 0.7, 0.5, 0.3, 0.15]),  # Lower min
]

print("\nGRID SEARCH: Sizing Thresholds")
print("="*90)
print(f"{'Thresholds':<35} {'Sizes':<25} {'Return':>10} {'Sharpe':>10}")
print("-"*90)

for thresholds, sizes in threshold_sets:
    func = make_stepped_sizing(thresholds, sizes)
    result = backtest_strat003(df, func, str(thresholds))
    grid_results.append(result)
    
    thresh_str = str(thresholds)
    sizes_str = str(sizes)
    print(f"{thresh_str:<35} {sizes_str:<25} {result['total_return']:>9.0f}% {result['sharpe']:>10.2f}")

best_grid = max(grid_results, key=lambda x: x['sharpe'])
print(f"\n🏆 Best: {best_grid['name']} (Sharpe: {best_grid['sharpe']:.2f})")

---
## Part 5: Walk-Forward Validation

In [ ]:
# Walk-forward: Train on first 70%, test on last 30%
split_idx = int(len(df) * 0.7)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

print(f"\nWALK-FORWARD VALIDATION")
print("="*80)
print(f"Train: {train_df.index[0].date()} to {train_df.index[-1].date()} ({len(train_df)} days)")
print(f"Test:  {test_df.index[0].date()} to {test_df.index[-1].date()} ({len(test_df)} days)")

# Test key strategies on both periods
key_strategies = [
    ('Fixed 100%', sizing_fixed),
    ('Stepped v1', sizing_stepped_v1),
    ('Linear', sizing_linear),
]

print(f"\n{'Strategy':<15} {'Train Sharpe':>15} {'Test Sharpe':>15} {'Overfit?':>12}")
print("-"*60)

for name, func in key_strategies:
    train_result = backtest_strat003(train_df, func, name)
    test_result = backtest_strat003(test_df, func, name)
    
    overfit = 'YES ❌' if test_result['sharpe'] < train_result['sharpe'] * 0.5 else 'NO ✅'
    
    print(f"{name:<15} {train_result['sharpe']:>15.2f} {test_result['sharpe']:>15.2f} {overfit:>12}")

---
## Part 6: Final v3 Specification

In [ ]:
print("\n" + "#"*80)
print("STRAT-003 v3 FINAL SPECIFICATION")
print("#"*80)

print("""
ENTRY:
  Trigger: STH-SOPR < 1.0 (first day)
  
  Position Size (based on Checkmate signal):
    signal <= -1.0:  100%  (very bullish - full conviction)
    signal <= -0.5:   80%
    signal <=  0.0:   60%
    signal <=  0.5:   40%
    signal >   0.5:   25%  (bearish - minimum exposure)

EXIT:
  Before trigger: 30% trailing stop from peak
  After trigger:  15% trailing stop from peak
  
  Trigger: MVRV > 2.5 AND LTH-SOPR > 1.5

RATIONALE:
  - STH-SOPR < 1 = weak hands capitulating (good entry)
  - Checkmate signal adds conviction layer:
      Low signal = market undervalued = go big
      High signal = market extended = go small
  - LTH-SOPR > 1.5 = smart money distributing (exit time)
""")

# Current recommendation
latest = df.iloc[-1]
current_size = sizing_stepped_v1(latest['checkmate'])

print(f"\nCURRENT STATE ({latest.name.date()}):")
print(f"  Price: ${latest['price']:,.0f}")
print(f"  STH-SOPR: {latest['sth_sopr']:.3f} {'⚠️ ENTRY SIGNAL' if latest['sth_sopr'] < 1 else ''}")
print(f"  Checkmate Signal: {latest['checkmate']:+.2f}")
print(f"  → Recommended Size: {current_size:.0%}")
print(f"\n  MVRV: {latest['mvrv']:.2f}")
print(f"  LTH-SOPR: {latest['lth_sopr']:.3f}")
print(f"  Exit Trigger: {'🔴 ACTIVE' if latest['exit_trigger'] else '⚪ Not active'}")

In [ ]:
# Final comparison chart
fig, ax = plt.subplots(figsize=(10, 6))

strategies = ['Fixed 100%', best['name']]
metrics = ['Return (%)', 'Sharpe', 'Max DD (%)', 'Win Rate (%)']

fixed = results[0]
data_fixed = [fixed['total_return']/100, fixed['sharpe'], -fixed['max_dd']/10, fixed['win_rate']/10]
data_best = [best['total_return']/100, best['sharpe'], -best['max_dd']/10, best['win_rate']/10]

x = np.arange(len(metrics))
width = 0.35

bars1 = ax.bar(x - width/2, data_fixed, width, label='Fixed 100%', color='gray')
bars2 = ax.bar(x + width/2, data_best, width, label=best['name'], color='#22c55e')

ax.set_ylabel('Value (scaled)')
ax.set_title('Fixed vs Signal-Based Sizing', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\n📊 Summary:")
print(f"  Return: {fixed['total_return']:.0f}% → {best['total_return']:.0f}% ({(best['total_return']/fixed['total_return']-1)*100:+.0f}%)")
print(f"  Sharpe: {fixed['sharpe']:.2f} → {best['sharpe']:.2f} ({(best['sharpe']/fixed['sharpe']-1)*100:+.0f}%)")
print(f"  MaxDD:  {fixed['max_dd']:.0f}% → {best['max_dd']:.0f}% ({(best['max_dd']/fixed['max_dd']-1)*100:+.0f}%)")